## A. Lord and Insert


In [34]:
import pandas as pd

students = pd.read_csv(
    "data/raw/students_messy.csv", na_values=["", "missing", "N/A", "-"]
)
print(students.head())
print(students.shape)
print(students.dtypes)
print(students.isna().sum())
print("Exact duplicates:", students.duplicated().sum())

   student_id         name program        city  attendance  assignment  \
0        1001     Ali Khan    BSAI   Islamabad        85.0        78.0   
1        1002   Sara Ahmed   BS AI   islamabad         NaN        88.0   
2        1003   Ahmed Raza    BSAI  Rawalpindi        92.0        80.0   
3        1003   Ahmed Raza    BSAI  Rawalpindi        92.0        80.0   
4        1004  Ayesha Noor    BSCS         Rwp       110.0        86.0   

   midterm  final admission_date   
0     72.0   81.0     2025-09-01   
1     68.0   75.0     01/09/2025   
2      NaN   88.0     2025/09/01   
3      NaN   88.0     2025/09/01   
4     75.0   90.0     09-01-2025   
(11, 9)
student_id           int64
name                   str
program                str
city                   str
attendance         float64
assignment         float64
midterm            float64
final              float64
admission_date         str
dtype: object
student_id         0
name               0
program            0
city       

## B1. Select and filter

In [35]:
perf_cols = [
    "student_id",
    "name",
    "attendance",
    "assignment",
    "midterm",
    "final",
]
print("1. Performance View:")
print(students[perf_cols].head())

ai_mask = students["program"].str.contains("AI", case=False, na=False)
print("\n2. Likely AI Students:")
print(students.loc[ai_mask, ["student_id", "name", "program"]])

print("\n3. Attendance < 75%:")
print(students[students["attendance"] < 75][["student_id", "name", "attendance"]])

print("\n4. Final Marks >= 80:")
print(students[students["final"] >= 80][["student_id", "name", "final"]])

sorted_students = students.sort_values(by="final", ascending=False, na_position="last")
print("\n5. Sorted by Final (Top 5):")
print(sorted_students[["student_id", "name", "final"]].head())

print("\n.loc vs .iloc:")
print(".loc (Label):", students.loc[0, "name"])
print(".iloc (Index):", students.iloc[0, 1])

1. Performance View:
   student_id         name  attendance  assignment  midterm  final
0        1001     Ali Khan        85.0        78.0     72.0   81.0
1        1002   Sara Ahmed         NaN        88.0     68.0   75.0
2        1003   Ahmed Raza        92.0        80.0      NaN   88.0
3        1003   Ahmed Raza        92.0        80.0      NaN   88.0
4        1004  Ayesha Noor       110.0        86.0     75.0   90.0

2. Likely AI Students:
   student_id          name  program
0        1001      Ali Khan     BSAI
1        1002    Sara Ahmed    BS AI
2        1003    Ahmed Raza     BSAI
3        1003    Ahmed Raza     BSAI
5        1005     Hamza Ali   BS-AI 
7        1007   Usman Tariq       AI
8        1008  Maryam Iqbal     BSAI

3. Attendance < 75%:
   student_id         name  attendance
5        1005    Hamza Ali        74.0
7        1007  Usman Tariq        69.0
9        1009  Bilal Ahmed        -5.0

4. Final Marks >= 80:
    student_id          name  final
0         1001      

## B2. Investigate duplicates

In [36]:
exact_dup_count = students.duplicated().sum()
print("Exact duplicate rows to remove:", exact_dup_count)
print(students[students.duplicated(keep=False)])

students = students.drop_duplicates().reset_index(drop=True)
print("\nShape after removing exact duplicates:", students.shape)


repeated_ids = students[students.duplicated(subset=["student_id"], keep=False)]
print("\nRepeated student IDs:")
print(repeated_ids[["student_id", "name", "program", "final"]])

Exact duplicate rows to remove: 1
   student_id        name program        city  attendance  assignment  \
2        1003  Ahmed Raza    BSAI  Rawalpindi        92.0        80.0   
3        1003  Ahmed Raza    BSAI  Rawalpindi        92.0        80.0   

   midterm  final admission_date   
2      NaN   88.0     2025/09/01   
3      NaN   88.0     2025/09/01   

Shape after removing exact duplicates: (10, 9)

Repeated student IDs:
Empty DataFrame
Columns: [student_id, name, program, final]
Index: []


## B3. Standardize text

In [37]:
students["program"] = students["program"].astype("string").str.strip()
students["city"] = students["city"].astype("string").str.strip()

students["program"] = students["program"].str.upper()
program_map = {
    "BS AI": "BSAI",
    "BS-AI": "BSAI",
    "AI": "BSAI",
    "BS DS": "BSDS",
    "BS-DS": "BSDS",
    "DS": "BSDS",
}
students["program"] = students["program"].replace(program_map)

students["city"] = students["city"].str.title()
students["city"] = students["city"].replace({"Rwp": "Rawalpindi"})
students["city"] = students["city"].fillna("Unknown")

print("Cleaned Program Values:", students["program"].unique())
print("Cleaned City Values:", students["city"].unique())

Cleaned Program Values: <StringArray>
['BSAI', 'BSCS', 'BSDS']
Length: 3, dtype: string
Cleaned City Values: <StringArray>
['Islamabad', 'Rawalpindi', 'Unknown']
Length: 3, dtype: string


## C1. Convert types

In [38]:

students.columns = students.columns.str.strip()

score_cols = ["attendance", "assignment", "midterm", "final"]
for col in score_cols:
    if col in students.columns:
        students[col] = pd.to_numeric(students[col], errors="coerce")

date_col = [c for c in students.columns if "date" in c.lower() or "admission" in c.lower()]
if date_col:
    target_date = date_col[0]
    students[target_date] = pd.to_datetime(
        students[target_date], dayfirst=True, errors="coerce"
    )

if "student_id" in students.columns:
    students["student_id"] = students["student_id"].astype(str)

print("Data Types after conversion:")
print(students.dtypes)
print("\nMissing values after coercion:")
print(students.isna().sum())

Data Types after conversion:
student_id                   str
name                         str
program                   string
city                      string
attendance               float64
assignment               float64
midterm                  float64
final                    float64
admission_date    datetime64[us]
dtype: object

Missing values after coercion:
student_id        0
name              0
program           0
city              0
attendance        1
assignment        1
midterm           1
final             1
admission_date    5
dtype: int64


## C2. Validate ranges (0 to 100)

In [39]:

cleaning_log = []


for col in score_cols:
    invalid_mask = (students[col] < 0) | (students[col] > 100)
    invalid_records = students[invalid_mask]

    for _, row in invalid_records.iterrows():
        cleaning_log.append(
            {
                "student_id": row["student_id"],
                "column": col,
                "original_value": row[col],
                "decision": "Replaced with NA",
                "reason": "Value outside valid range 0-100",
            }
        )

    
    students.loc[invalid_mask, col] = pd.NA


cleaning_log_df = pd.DataFrame(cleaning_log)
print("Cleaning Log (Out of range records):")
print(cleaning_log_df)


out_of_range_check = sum(
    ((students[col] < 0) | (students[col] > 100)).sum() for col in score_cols
)
print("\nRemaining out-of-range values count:", out_of_range_check)

Cleaning Log (Out of range records):
  student_id      column  original_value          decision  \
0       1004  attendance           110.0  Replaced with NA   
1       1009  attendance            -5.0  Replaced with NA   

                            reason  
0  Value outside valid range 0-100  
1  Value outside valid range 0-100  

Remaining out-of-range values count: 0


## C3. Handle missing values

In [40]:

missing_before = students[score_cols].isna().sum()
print("Missing counts before imputation:\n", missing_before)

for col in score_cols:
    students[f"{col}_imputed"] = students[col].isna()

overall_att_median = students["attendance"].median()
students["attendance"] = students.groupby("program")[
    "attendance"
].transform(lambda group: group.fillna(group.median()))
students["attendance"] = students["attendance"].fillna(overall_att_median)

for col in ["assignment", "midterm", "final"]:
    col_median = students[col].median()
    students[col] = students[col].fillna(col_median)
    print(f"Filled missing {col} with median: {col_median}")

missing_after = students[score_cols].isna().sum()
print("\nMissing counts after imputation:\n", missing_after)

assert (
    missing_after.sum() == 0
), "Scoring columns mein abhi bhi missing values hain!"
print(
    "\nVerification Passed: All scoring columns filled successfully (0 missing)."
)

Missing counts before imputation:
 attendance    3
assignment    1
midterm       1
final         1
dtype: int64
Filled missing assignment with median: 80.0
Filled missing midterm with median: 72.0
Filled missing final with median: 81.0

Missing counts after imputation:
 attendance    0
assignment    0
midterm       0
final         0
dtype: int64

Verification Passed: All scoring columns filled successfully (0 missing).


## D1. Create features

In [41]:
import numpy as np

students["weighted_score"] = (
    students["assignment"] * 0.20
    + students["midterm"] * 0.30
    + students["final"] * 0.50
)

attendance_conditions = [
    students["attendance"] < 75,
    (students["attendance"] >= 75) & (students["attendance"] < 85),
    students["attendance"] >= 85,
]
attendance_choices = ["Low", "Satisfactory", "Excellent"]
students["attendance_status"] = np.select(
    attendance_conditions, attendance_choices, default="Unknown"
)

grade_conditions = [
    students["weighted_score"] >= 85,
    students["weighted_score"] >= 75,
    students["weighted_score"] >= 65,
    students["weighted_score"] >= 50,
]
grade_choices = ["A", "B", "C", "D"]
students["grade"] = np.select(grade_conditions, grade_choices, default="F")

students["at_risk"] = (students["attendance"] < 75) | (
    students["weighted_score"] < 50
)

assert students["weighted_score"].between(0, 100).all(), "Weighted score out of bounds!"

print("D1 Features Sample:")
print(students[["student_id", "weighted_score", "attendance_status", "grade", "at_risk"]].head())

D1 Features Sample:
  student_id  weighted_score attendance_status grade  at_risk
0       1001            77.7         Excellent     B    False
1       1002            75.5         Excellent     B    False
2       1003            81.6         Excellent     B    False
3       1004            84.7      Satisfactory     B    False
4       1005            70.1               Low     C     True


## D2. Group and summarize

In [42]:

program_summary = (
    students.groupby("program")
    .agg(student_count=("student_id", "count"), avg_score=("weighted_score", "mean"))
    .reset_index()
)
print("Program Summary:")
print(program_summary)
print()

grade_distribution = (
    students["grade"].value_counts().reset_index()
)
grade_distribution.columns = ["grade", "count"]
print("Grade Distribution:")
print(grade_distribution)
print()

total_students = len(students)
assert program_summary["student_count"].sum() == total_students
assert grade_distribution["count"].sum() == total_students

print("Verification Passed: Both counts match total student count!")

Program Summary:
  program  student_count  avg_score
0    BSAI              6      74.95
1    BSCS              2      66.40
2    BSDS              2      83.75

Grade Distribution:
  grade  count
0     B      5
1     A      2
2     C      1
3     D      1
4     F      1

Verification Passed: Both counts match total student count!


 ## Part E — Save, report, and commit

In [43]:
import os
import pandas as pd


os.makedirs("Data/processed", exist_ok=True)
os.makedirs("Reports", exist_ok=True)

assert students["student_id"].is_unique, "Student IDs are not unique!"
assert students["student_id"].isna().sum() == 0, "Missing student IDs detected!"

score_check_cols = ["attendance", "assignment", "midterm", "final", "weighted_score"]
for col in score_check_cols:
    assert students[col].between(0, 100).all(), f"{col} contains values outside 0-100!"

print("All verifications passed successfully!")

students.to_csv("Data/processed/students_clean.csv", index=False)
program_summary.to_csv("Data/processed/program_summary.csv", index=False)
grade_distribution.to_csv("Data/processed/grade_distribution.csv", index=False)


cleaning_log_df.to_csv("Reports/cleaning_log.csv", index=False)

if "invalid_records" in locals() and len(invalid_records) > 0:
    invalid_records.to_csv("Reports/invalid_records.csv", index=False)
else:
    cleaning_log_df.to_csv("Reports/invalid_records.csv", index=False)


reloaded = pd.read_csv("Data/processed/students_clean.csv")
print("Reloaded Shape:", reloaded.shape)
print("All files saved and verified successfully!")

All verifications passed successfully!
Reloaded Shape: (10, 17)
All files saved and verified successfully!
